In [11]:
import ee
from lib.utils.ee.ccdc_utils import build_segment_tag, filter_coefs, build_ccd_image, get_multi_coefs
from pprint import pprint


ee.Initialize()

# Settings

STABLE_POINTS = 5
UNSTABLE_POINTS = 10

START_YEAR = 2016
END_YEAR   = 2018

AOI = [
    (-65.00330743296635, -9.986381612112117),
    (-65.00330743296635, -14.993821533992973),
    (-59.98255547984135, -14.993821533992973),
    (-59.98255547984135, -9.986381612112117),
]

aoi = ee.Geometry.Polygon(AOI)

bands = ['SWIR1']
coefficients = ["INTP", "SLP", "COS", "SIN", "COS2", "SIN2", "COS3", "SIN3", "RMSE"]
segments = build_segment_tag(10)

ccdc_raw = ee.ImageCollection("GOOGLE/GLOBAL_CCDC/V1").mosaic()

def mid_decimal_year(y):
    return ee.Number(y).add(0.5)

def segment_bounds(ccdc_image, date):
    start = filter_coefs(ccdc_image, date, "", "tStart", segments, "before")
    end = filter_coefs(ccdc_image, date, "", "tEnd", segments, "before")
    return start.rename("segment_tStart").addBands(end.rename("segment_tEnd"))


n_segments = segments.size().getInfo()
ccdc_image = build_ccd_image(ccdc_raw, n_segments, bands)

start_names = segments.map(lambda s: ee.String(s).cat("_tStart"))
end_names   = segments.map(lambda s: ee.String(s).cat("_tEnd"))
all_names   = start_names.cat(end_names)

segment_image = ccdc_image.select([".*_tStart", ".*_tEnd"]).rename(all_names)

coefs_start = get_multi_coefs(
    ccdc_image, mid_decimal_year(START_YEAR), bands, coefficients, True, segments, "before"
).clip(aoi)

coefs_end = get_multi_coefs(
    ccdc_image, mid_decimal_year(END_YEAR), bands, coefficients, True, segments, "before"
).clip(aoi)

delta = coefs_end.subtract(coefs_start).abs()
diffMag = delta.reduce(ee.Reducer.max())

noDiffMask = diffMag.eq(0)
diffMask   = diffMag.gt(0)

classImage = ee.Image(0).where(diffMask, 1).rename('class')

validMask = coefs_start.mask().reduce(ee.Reducer.allNonZero()) \
    .And(coefs_end.mask().reduce(ee.Reducer.allNonZero()))


seg_start = segment_bounds(ccdc_image, mid_decimal_year(START_YEAR)).rename(["startSeg_tStart", "startSeg_tEnd"])
seg_end   = segment_bounds(ccdc_image, mid_decimal_year(END_YEAR)).rename(["endSeg_tStart", "endSeg_tEnd"])

sampleImg = (
    classImage
        .addBands(diffMag.rename("diffMag"))
        .addBands(segment_image)
)

allPoints = sampleImg.sample(
    region=aoi,
    scale=1000,
    geometries=True,
    seed=42
)

sortedPoints = allPoints.sort('diffMag', False)  # False = descending order

stablePoints = sortedPoints.filter(ee.Filter.eq('class', 0)).limit(STABLE_POINTS)
unstablePoints = sortedPoints.filter(ee.Filter.eq('class', 1)).limit(UNSTABLE_POINTS)

sampledPoints = stablePoints.merge(unstablePoints)

def add_props(f):
    coords = f.geometry().coordinates()
    status = ee.String(ee.Algorithms.If(ee.Number(f.get('class')).eq(1), 'changed', 'unchanged'))
    return f.set({
        'lon': coords.get(0),
        'lat': coords.get(1),
        'START_YEAR': START_YEAR,
        'END_YEAR': END_YEAR,
        'status': status,
        'segments': segments
    })

samples_tagged = sampledPoints.map(add_props)

features = samples_tagged.getInfo()['features']

pprint(features[0])


# selectors = ['class', 'status', 'lon', 'lat', 'START_YEAR', 'END_YEAR', 'diffMag']
# task = ee.batch.Export.table.toDrive(
#     collection=samples_tagged,
#     description=f'ccdc_samples_{START_YEAR}_{END_YEAR}',
#     fileNamePrefix=f'ccdc_samples_{START_YEAR}_{END_YEAR}',
#     fileFormat='CSV',
#     selectors=selectors
# )
# task.start()

# print('Export started:', f'ccdc_samples_{START_YEAR}_{END_YEAR}')

{'geometry': {'coordinates': [-64.35081537790192, -14.997373668375412],
              'geodesic': False,
              'type': 'Point'},
 'id': '1_0',
 'properties': {'END_YEAR': 2018,
                'S10_tEnd': 0,
                'S10_tStart': 0,
                'S1_tEnd': 2005.046142578125,
                'S1_tStart': 1999.526611328125,
                'S2_tEnd': 2019.83056640625,
                'S2_tStart': 2005.26513671875,
                'S3_tEnd': 0,
                'S3_tStart': 0,
                'S4_tEnd': 0,
                'S4_tStart': 0,
                'S5_tEnd': 0,
                'S5_tStart': 0,
                'S6_tEnd': 0,
                'S6_tStart': 0,
                'S7_tEnd': 0,
                'S7_tStart': 0,
                'S8_tEnd': 0,
                'S8_tStart': 0,
                'S9_tEnd': 0,
                'S9_tStart': 0,
                'START_YEAR': 2016,
                'class': 0,
                'diffMag': 0,
                'lat': -14.9973736683

In [12]:
import json
import os
from datetime import datetime

samples_tagged_info = samples_tagged.getInfo()
features = samples_tagged_info['features']

points_list = []
for feature in samples_tagged_info['features']:
    coords = feature['geometry']['coordinates']
    points_list.append(coords)

output_data = {"points": points_list}

current_datetime_str = datetime.now().strftime("%Y%m%d_%H%M%S")
total_points_count = len(points_list)
group_dir_name = f"{current_datetime_str}-{total_points_count}_points"
output_base_dir = os.path.join("lib", "point_groups", "groups")
output_dir = os.path.join(output_base_dir, group_dir_name)
output_filepath = os.path.join(output_dir, "points.json")

os.makedirs(output_dir, exist_ok=True)

with open(output_filepath, 'w') as f:
    json.dump(output_data, f, indent=2)

print(f"Successfully saved {len(points_list)} points to: {output_filepath}")


Successfully saved 15 points to: lib/point_groups/groups/20251004_105055-15_points/points.json


In [14]:
import json
import os

import ee
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lib.constants import (
    Kalman,
    DATE_LABEL,
    TIMESTAMP_LABEL,
    CCDC,
    Index,
    Sensor,
)
from kalman.kalman_module import append_ccdc_coefficients
from lib.study_packages import get_collection
from lib.utils import utils

if 'output_dir' not in globals():
    raise RuntimeError('Run the point export cell before generating time series outputs.')

years = list(range(START_YEAR, END_YEAR + 1))

collection_parameters = {
    'index': Index.SWIR,
    'sensors': [Sensor.L7, Sensor.L8, Sensor.L9],
    'day_step_size': 6,
    'start_doy': 1,
    'end_doy': 365,
    'cloud_cover_threshold': 20,
}

plot_dir = os.path.join(output_dir, 'plots')
os.makedirs(plot_dir, exist_ok=True)

timeseries_frames = []
feature_metadata = []
ccdc_columns = set()

for feature_index, feature in enumerate(features):
    coords = feature['geometry']['coordinates']
    props = feature.get('properties', {})
    feature_metadata.append({
        'id': feature_index,
        'coordinates': coords,
        **props,
    })
    for year in years:
        ee_collection = get_collection(**{
            **collection_parameters,
            'years': [year],
            'study_area': ee.Geometry.Point(coords),
        })
        if ee_collection.size().getInfo() == 0:
            continue
        first_image = ee_collection.first()
        band_names = first_image.bandNames().getInfo()
        if not band_names:
            continue
        measurement_band = band_names[0]

        def add_measurement_and_timestamp(image):
            measurement = image.select(measurement_band).rename(Kalman.Z.value)
            timestamp = ee.Image(image.date().millis()).rename(TIMESTAMP_LABEL)
            return image.addBands([measurement, timestamp])

        augmented_collection = (
            ee_collection
            .map(append_ccdc_coefficients)
            .map(add_measurement_and_timestamp)
        )

        augmented_first = augmented_collection.first()
        augmented_band_names = augmented_first.bandNames().getInfo() if augmented_first else []
        if not augmented_band_names:
            continue

        values = utils.get_image_collection_pixels(coords, augmented_collection)
        if values.size == 0:
            continue

        rows = values.size // len(augmented_band_names)
        if rows == 0:
            continue
        data = values.reshape(rows, len(augmented_band_names))
        yearly_df = pd.DataFrame(data, columns=augmented_band_names)

        yearly_df[DATE_LABEL] = (
            pd.to_datetime(yearly_df[TIMESTAMP_LABEL], unit='ms')
            .dt.strftime('%Y-%m-%d')
        )
        yearly_df = yearly_df.assign(
            feature_index=feature_index,
            lon=coords[0],
            lat=coords[1],
        )
        for key in ('status', 'class'):
            if key in props:
                yearly_df[key] = props[key]

        ccdc_cols = [
            col for col in augmented_band_names
            if col.startswith(f"{CCDC.BAND_PREFIX.value}_")
        ]
        if CCDC.FIT.value in augmented_band_names:
            ccdc_cols.append(CCDC.FIT.value)
        ccdc_columns.update(ccdc_cols)

        timeseries_frames.append(yearly_df)

if not timeseries_frames:
    raise RuntimeError('No time series data available for the sampled features.')

combined_df = pd.concat(timeseries_frames, ignore_index=True)

ordered_ccdc_columns = sorted(ccdc_columns)
selected_columns = [
    'feature_index',
    'lon',
    'lat',
    DATE_LABEL,
    Kalman.Z.value,
    *ordered_ccdc_columns,
]
if 'status' in combined_df.columns:
    selected_columns.append('status')
if 'class' in combined_df.columns and 'class' not in selected_columns:
    selected_columns.append('class')

timeseries_output = combined_df.loc[:, selected_columns].copy()
numeric_columns = [
    col for col in [Kalman.Z.value, *ordered_ccdc_columns]
    if col in timeseries_output.columns
]
timeseries_output[numeric_columns] = timeseries_output[numeric_columns].apply(
    pd.to_numeric, errors='coerce'
)
timeseries_output.sort_values(['feature_index', DATE_LABEL], inplace=True)

timeseries_output_path = os.path.join(output_dir, 'feature_timeseries.csv')
timeseries_output.to_csv(timeseries_output_path, index=False)

for feature_index, feature_df in timeseries_output.groupby('feature_index'):
    plot_df = feature_df[feature_df[Kalman.Z.value] != 0].copy()
    status_value = 'unknown'
    if 'status' in feature_df.columns:
        non_na_status = feature_df['status'].dropna()
        if not non_na_status.empty:
            status_value = str(non_na_status.iloc[0])
    status_slug = status_value.lower().replace(' ', '_') if status_value else 'unknown'

    fig, ax = plt.subplots(figsize=(10, 5))
    if not plot_df.empty:
        dates_measurement = pd.to_datetime(plot_df[DATE_LABEL])
        ax.scatter(dates_measurement, plot_df[Kalman.Z.value], s=20, alpha=0.8, label='Measurement')

    if CCDC.FIT.value in feature_df.columns:
        dates_ccdc = pd.to_datetime(feature_df[DATE_LABEL])
        ax.plot(
            dates_ccdc,
            feature_df[CCDC.FIT.value],
            color='tab:orange',
            linewidth=1.5,
            label='CCDC fit',
        )

    ax.set_title(f'Point {feature_index} ({status_value})')
    ax.set_xlabel('Date')
    ax.set_ylabel('Value')
    ax.legend()
    fig.autofmt_xdate()

    plot_path = os.path.join(plot_dir, f'point_{feature_index}_{status_slug}.png')
    fig.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close(fig)

feature_metadata_path = os.path.join(output_dir, 'feature_metadata.json')
with open(feature_metadata_path, 'w') as f:
    json.dump({'features': feature_metadata}, f, indent=2)

print(f'Saved time series to: {timeseries_output_path}')
print(f'Saved feature metadata to: {feature_metadata_path}')
print(f'Saved plots to: {plot_dir}')


Saved time series to: lib/point_groups/groups/20251004_105055-15_points/feature_timeseries.csv
Saved feature metadata to: lib/point_groups/groups/20251004_105055-15_points/feature_metadata.json
Saved plots to: lib/point_groups/groups/20251004_105055-15_points/plots
